In [1]:
# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# import os

# # ── font sizes ────────────────────────────────────────────────────────────────
# FONT_SIZE_AXIS_LABEL = 11
# FONT_SIZE_TICK_LABEL = 10
# FONT_SIZE_LEGEND     = 9.5
# FONT_SIZE_TITLE      = 12

# # ── LaTeX rendering ───────────────────────────────────────────────────────────
# plt.rcParams.update({
#     "text.usetex":      True,
#     "font.family":      "serif",
#     "font.serif":       ["Computer Modern Roman"],
#     "axes.labelsize":   FONT_SIZE_AXIS_LABEL,
#     "xtick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "ytick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "legend.fontsize":  FONT_SIZE_LEGEND,
#     "axes.titlesize":   FONT_SIZE_TITLE,
# })

# # ── case definitions ──────────────────────────────────────────────────────────
# BASE = r"D:\FCAI\effusion_coupon_CompundAngle\infared_data"
# SUB  = (r"time_average\average_temperature\half_domain_temperature"
#         r"\summary_avg_std\cropped_end_30mm_centre")
# CSV  = "combined_spanwise_avg_std_cropped.csv"

# OUT_DIR  = r"D:\FCAI\effusion_coupon_CompundAngle\data_processing\spanwise_averaging"
# OUT_FILE = os.path.join(OUT_DIR, "spanwise_temperature.svg")

# phis     = [0.9, 1.0, 1.18]
# phi_tags = {"0.9": "09", "1.0": "10", "1.18": "118"}
# crs      = [500, 625, 860, 1125]

# COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]

# # ── helpers ───────────────────────────────────────────────────────────────────
# def load_csv(cr, phi_str):
#     folder = f"cr_{cr}_phi_{phi_tags[phi_str]}"
#     path = os.path.join(BASE, folder, SUB, CSV)
#     if not os.path.isfile(path):
#         print(f"  [WARNING] not found: {path}")
#         return None
#     df = pd.read_csv(path)
#     df.columns = ["x", "avg", "std"]
#     return df

# def detect_jumps(x, threshold_factor=5):
#     dx = np.diff(x)
#     med = np.median(dx)
#     return np.concatenate(([False], dx > threshold_factor * med))

# def split_segments(df):
#     x = df["x"].values
#     jump = detect_jumps(x)
#     segments, start = [], 0
#     for i in range(1, len(x)):
#         if jump[i]:
#             segments.append(df.iloc[start:i].copy())
#             start = i
#     segments.append(df.iloc[start:].copy())
#     return segments

# # ── load data ─────────────────────────────────────────────────────────────────
# data, missing = {}, []
# for phi in phis:
#     phi_str = str(phi)
#     for cr in crs:
#         df = load_csv(cr, phi_str)
#         if df is None:
#             missing.append((cr, phi))
#         data[(cr, phi)] = df

# if missing:
#     print(f"\n[INFO] {len(missing)} case(s) missing -- curves will be skipped: {missing}\n")

# # ── collect global y-range across all subplots for shared tick positions ──────
# y_all_min, y_all_max = np.inf, -np.inf
# for phi in phis:
#     for cr in crs:
#         df = data.get((cr, phi))
#         if df is None:
#             continue
#         y_all_min = min(y_all_min, (df["avg"] - df["std"]).min())
#         y_all_max = max(y_all_max, (df["avg"] + df["std"]).max())

# # 6 evenly-spaced tick positions spanning the global data range
# N_TICKS   = 6
# y_ticks   = np.linspace(y_all_min, y_all_max, N_TICKS)
# # normalised positions in [0, 1] — same relative locations on every subplot
# y_norm    = np.linspace(0.0, 1.0, N_TICKS)

# # ── plotting ──────────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)
# fig.subplots_adjust(left=0.07, right=0.97, top=0.87, bottom=0.10, wspace=0.32)

# for ax_idx, (ax, phi) in enumerate(zip(axes, phis)):

#     # ── draw curves ───────────────────────────────────────────────────────────
#     for ci, cr in enumerate(crs):
#         df = data.get((cr, phi))
#         if df is None:
#             continue

#         color    = COLORS[ci]
#         label    = r"$\dot{m}_c = $" + f"${cr}$" + r"$~\mathrm{slpm}$"
#         segments = split_segments(df)

#         for seg_i, seg in enumerate(segments):
#             x   = seg["x"].values * 1e3
#             avg = seg["avg"].values
#             std = seg["std"].values
#             lbl = label if seg_i == 0 else "_nolegend_"
#             ax.plot(x, avg, color=color, lw=1.6, label=lbl)
#             ax.fill_between(x, avg - std, avg + std,
#                             color=color, alpha=0.18, linewidth=0)

#     # ── per-subplot y-axis: 6 ticks at the same relative positions ───────────
#     y_lo, y_hi = ax.get_ylim()
#     # map the shared normalised positions into this subplot's actual y-range
#     ax_ticks = y_lo + y_norm * (y_hi - y_lo)
#     ax.set_yticks(ax_ticks)
#     # format tick labels to 0 decimal places (integers look cleaner)
#     ax.yaxis.set_major_formatter(ticker.FormatStrFormatter(r"$%.0f$"))
#     ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

#     ax.set_title(r"$\phi = " + f"{phi}$", fontsize=FONT_SIZE_TITLE, pad=7)
#     ax.set_xlabel(r"$x~\mathrm{(mm)}$", fontsize=FONT_SIZE_AXIS_LABEL)
#     if ax_idx == 0:
#         ax.set_ylabel(r"Temperature $\mathrm{(^{\circ}C)}$",
#                       fontsize=FONT_SIZE_AXIS_LABEL)

#     ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
#     ax.tick_params(which="both", direction="in", top=True, right=True, length=4)
#     ax.tick_params(which="minor", length=2)
#     ax.grid(which="major", ls="--", lw=0.4, alpha=0.45)
#     ax.spines[["top", "right"]].set_linewidth(0.8)

#     # ── 2×2 legend in top-right corner of each subplot ───────────────────────
#     handles, labels = ax.get_legend_handles_labels()
#     ax.legend(handles, labels,
#               loc="upper right",
#               ncol=2,
#               fontsize=FONT_SIZE_LEGEND,
#               frameon=True,
#               framealpha=0.9,
#               edgecolor="0.7",
#               handlelength=1.8,
#               handletextpad=0.5,
#               columnspacing=0.8,
#               borderpad=0.5)

# fig.suptitle(
#     r"Spanwise-Averaged Temperature --- Effusion Coupon (Compound Angle)",
#     fontsize=FONT_SIZE_TITLE, y=0.97
# )

# # ── save ──────────────────────────────────────────────────────────────────────
# os.makedirs(OUT_DIR, exist_ok=True)
# fig.savefig(OUT_FILE, format="svg", bbox_inches="tight")
# print(f"Figure saved -> {OUT_FILE}")
# plt.show()

In [2]:
# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# import matplotlib.patches as mpatches
# import os

# # ── font sizes ────────────────────────────────────────────────────────────────
# FONT_SIZE_AXIS_LABEL = 12
# FONT_SIZE_TICK_LABEL = 13
# FONT_SIZE_LEGEND     = 12
# FONT_SIZE_TITLE      = 13

# # ── LaTeX rendering ───────────────────────────────────────────────────────────
# plt.rcParams.update({
#     "text.usetex":      True,
#     "font.family":      "serif",
#     "font.serif":       ["Computer Modern Roman"],
#     "axes.labelsize":   FONT_SIZE_AXIS_LABEL,
#     "xtick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "ytick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "legend.fontsize":  FONT_SIZE_LEGEND,
#     "axes.titlesize":   FONT_SIZE_TITLE,
# })

# # ── case definitions ──────────────────────────────────────────────────────────
# BASE = r"D:\FCAI\effusion_coupon_CompundAngle\infared_data"
# SUB  = (r"time_average\average_temperature\half_domain_temperature"
#         r"\summary_avg_std\cropped_end_30mm_centre")
# CSV  = "combined_spanwise_avg_std_cropped.csv"

# OUT_DIR  = r"D:\FCAI\effusion_coupon_CompundAngle\data_processing\spanwise_averaging"
# OUT_FILE = os.path.join(OUT_DIR, "spanwise_temperature_sharedy.svg")

# phis     = [0.9, 1.0, 1.18]
# phi_tags = {"0.9": "09", "1.0": "10", "1.18": "118"}
# crs      = [500, 625, 860, 1125]

# COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]

# # ── helpers ───────────────────────────────────────────────────────────────────
# def load_csv(cr, phi_str):
#     folder = f"cr_{cr}_phi_{phi_tags[phi_str]}"
#     path = os.path.join(BASE, folder, SUB, CSV)
#     if not os.path.isfile(path):
#         print(f"  [WARNING] not found: {path}")
#         return None
#     df = pd.read_csv(path)
#     df.columns = ["x", "avg", "std"]
#     return df

# def detect_jumps(x, threshold_factor=5):
#     dx = np.diff(x)
#     med = np.median(dx)
#     return np.concatenate(([False], dx > threshold_factor * med))

# def split_segments(df):
#     x = df["x"].values
#     jump = detect_jumps(x)
#     segments, start = [], 0
#     for i in range(1, len(x)):
#         if jump[i]:
#             segments.append(df.iloc[start:i].copy())
#             start = i
#     segments.append(df.iloc[start:].copy())
#     return segments

# def nice_ticks(y_lo, y_hi, n=6):
#     raw_step  = (y_hi - y_lo) / (n - 1)
#     magnitude = 10 ** np.floor(np.log10(raw_step))
#     nice_step = np.ceil(raw_step / (5 * magnitude)) * 5 * magnitude
#     t0    = np.floor(y_lo / nice_step) * nice_step
#     ticks = t0 + np.arange(n) * nice_step
#     return ticks, nice_step

# # ── load all data ─────────────────────────────────────────────────────────────
# data, missing = {}, []
# for phi in phis:
#     phi_str = str(phi)
#     for cr in crs:
#         df = load_csv(cr, phi_str)
#         if df is None:
#             missing.append((cr, phi))
#         data[(cr, phi)] = df

# if missing:
#     print(f"\n[INFO] {len(missing)} case(s) missing -- curves will be skipped: {missing}\n")

# # ── global y-range ────────────────────────────────────────────────────────────
# y_all_min, y_all_max = np.inf, -np.inf
# for phi in phis:
#     for cr in crs:
#         df = data.get((cr, phi))
#         if df is None:
#             continue
#         y_all_min = min(y_all_min, (df["avg"] - df["std"]).min())
#         y_all_max = max(y_all_max, (df["avg"] + df["std"]).max())

# y_ticks, step = nice_ticks(y_all_min, y_all_max, n=6)
# Y_LIM = (y_ticks[0] - 0.3 * step, y_ticks[-1] + 0.3 * step)

# # ── plotting ──────────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)
# fig.subplots_adjust(left=0.07, right=0.97, top=0.87, bottom=0.10, wspace=0.28)

# for ax_idx, (ax, phi) in enumerate(zip(axes, phis)):

#     # ── draw curves ───────────────────────────────────────────────────────────
#     for ci, cr in enumerate(crs):
#         df = data.get((cr, phi))
#         if df is None:
#             continue

#         color    = COLORS[ci]
#         label    = r"$\dot{m}_c = $" + f"${cr}$" + r"$~\mathrm{slpm}$"
#         segments = split_segments(df)

#         for seg_i, seg in enumerate(segments):
#             x   = seg["x"].values * 1e3
#             avg = seg["avg"].values
#             std = seg["std"].values
#             lbl = label if seg_i == 0 else "_nolegend_"
#             ax.plot(x, avg, color=color, lw=1.6, label=lbl)
#             ax.fill_between(x, avg - std, avg + std,
#                             color=color, alpha=0.18, linewidth=0)

#     # ── y-axis ────────────────────────────────────────────────────────────────
#     ax.set_ylim(Y_LIM)
#     ax.set_yticks(y_ticks)
#     ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f"{val:.0f}"))
#     ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

#     ax.set_title(r"$\phi = " + f"{phi}$", fontsize=FONT_SIZE_TITLE, pad=7)
#     ax.set_xlabel(r"$x~\mathrm{(mm)}$", fontsize=FONT_SIZE_AXIS_LABEL)
#     if ax_idx == 0:
#         ax.set_ylabel(r"Temperature $\mathrm{(^{\circ}C)}$",
#                       fontsize=FONT_SIZE_AXIS_LABEL)

#     ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
#     ax.tick_params(which="both", direction="in", top=True, right=True, length=4)
#     ax.tick_params(which="minor", length=2)
#     # ax.grid(which="major", ls="--", lw=0.4, alpha=0.45)
#     ax.spines[["top", "right"]].set_linewidth(0.8)

#     # ── legend: 4 cooling-rate entries + 1 std-band entry ────────────────────
#     handles, labels = ax.get_legend_handles_labels()

#     std_patch = mpatches.Patch(
#         facecolor="grey", alpha=0.35, edgecolor="none",
#         label=r"$\pm\,\sigma$"
#     )
#     handles.append(std_patch)
#     labels.append(r"$\pm\,\sigma$")

#     ax.legend(handles, labels,
#               loc="upper right",
#               ncol=2,
#               fontsize=FONT_SIZE_LEGEND,
#               frameon=False,
#               handlelength=1.6,
#               handletextpad=0.5,
#               columnspacing=0.8,
#               borderpad=0.5)

# # fig.suptitle(
# #     r"Spanwise-Averaged Temperature --- Effusion Coupon (Compound Angle)",
# #     fontsize=FONT_SIZE_TITLE, y=0.97
# # )

# # ── save ──────────────────────────────────────────────────────────────────────
# os.makedirs(OUT_DIR, exist_ok=True)
# fig.savefig(OUT_FILE, format="svg", bbox_inches="tight")
# print(f"Figure saved -> {OUT_FILE}")
# plt.show()

In [3]:
# ## only plot 3 equivalence ratio 

# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker
# import os

# # ── font sizes ────────────────────────────────────────────────────────────────
# FONT_SIZE_AXIS_LABEL = 12
# FONT_SIZE_TICK_LABEL = 13
# FONT_SIZE_LEGEND     = 12
# FONT_SIZE_TITLE      = 13

# # ── LaTeX rendering ───────────────────────────────────────────────────────────
# plt.rcParams.update({
#     "text.usetex":      True,
#     "font.family":      "serif",
#     "font.serif":       ["Computer Modern Roman"],
#     "axes.labelsize":   FONT_SIZE_AXIS_LABEL,
#     "xtick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "ytick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "legend.fontsize":  FONT_SIZE_LEGEND,
#     "axes.titlesize":   FONT_SIZE_TITLE,
# })

# # ── case definitions ──────────────────────────────────────────────────────────
# BASE = r"D:\FCAI\effusion_coupon_CompundAngle\infared_data"
# SUB  = (r"time_average\average_temperature\half_domain_temperature"
#         r"\summary_avg_std\cropped_end_30mm_centre")
# CSV  = "combined_spanwise_avg_std_cropped.csv"

# OUT_DIR  = r"D:\FCAI\effusion_coupon_CompundAngle\data_processing\spanwise_averaging"
# OUT_FILE = os.path.join(OUT_DIR, "spanwise_temperature_sharedy.svg")

# phis     = [0.9, 1.0, 1.18]
# phi_tags = {"0.9": "09", "1.0": "10", "1.18": "118"}
# crs      = [500, 625, 860, 1125]

# COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]

# # ── helpers ───────────────────────────────────────────────────────────────────
# def load_csv(cr, phi_str):
#     folder = f"cr_{cr}_phi_{phi_tags[phi_str]}"
#     path = os.path.join(BASE, folder, SUB, CSV)
#     if not os.path.isfile(path):
#         print(f"  [WARNING] not found: {path}")
#         return None
#     df = pd.read_csv(path)
#     df.columns = ["x", "avg", "std"]
#     return df

# def detect_jumps(x, threshold_factor=5):
#     dx = np.diff(x)
#     med = np.median(dx)
#     return np.concatenate(([False], dx > threshold_factor * med))

# def split_segments(df):
#     x = df["x"].values
#     jump = detect_jumps(x)
#     segments, start = [], 0
#     for i in range(1, len(x)):
#         if jump[i]:
#             segments.append(df.iloc[start:i].copy())
#             start = i
#     segments.append(df.iloc[start:].copy())
#     return segments

# def nice_ticks(y_lo, y_hi, n=6):
#     raw_step  = (y_hi - y_lo) / (n - 1)
#     magnitude = 10 ** np.floor(np.log10(raw_step))
#     nice_step = np.ceil(raw_step / (5 * magnitude)) * 5 * magnitude
#     t0    = np.floor(y_lo / nice_step) * nice_step
#     ticks = t0 + np.arange(n) * nice_step
#     return ticks, nice_step

# # ── load all data ─────────────────────────────────────────────────────────────
# data, missing = {}, []
# for phi in phis:
#     phi_str = str(phi)
#     for cr in crs:
#         df = load_csv(cr, phi_str)
#         if df is None:
#             missing.append((cr, phi))
#         data[(cr, phi)] = df

# if missing:
#     print(f"\n[INFO] {len(missing)} case(s) missing -- curves will be skipped: {missing}\n")

# # ── global y-range ────────────────────────────────────────────────────────────
# y_all_min, y_all_max = np.inf, -np.inf
# for phi in phis:
#     for cr in crs:
#         df = data.get((cr, phi))
#         if df is None:
#             continue
#         y_all_min = min(y_all_min, (df["avg"] - df["std"]).min())
#         y_all_max = max(y_all_max, (df["avg"] + df["std"]).max())

# y_ticks, step = nice_ticks(y_all_min, y_all_max, n=6)
# Y_LIM = (y_ticks[0] - 0.3 * step, y_ticks[-1] + 0.3 * step)

# # ── plotting ──────────────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), sharey=False)
# fig.subplots_adjust(left=0.07, right=0.97, top=0.87, bottom=0.10, wspace=0.28)

# for ax_idx, (ax, phi) in enumerate(zip(axes, phis)):

#     # ── draw curves ───────────────────────────────────────────────────────────
#     for ci, cr in enumerate(crs):
#         df = data.get((cr, phi))
#         if df is None:
#             continue

#         color    = COLORS[ci]
#         label    = r"$\dot{m}_c = $" + f"${cr}$" + r"$~\mathrm{slpm}$"
#         segments = split_segments(df)

#         for seg_i, seg in enumerate(segments):
#             x   = seg["x"].values * 1e3
#             avg = seg["avg"].values
#             std = seg["std"].values
#             lbl = label if seg_i == 0 else "_nolegend_"
#             ax.plot(x, avg, color=color, lw=1.6, label=lbl)

#     # ── y-axis ────────────────────────────────────────────────────────────────
#     ax.set_ylim(Y_LIM)
#     ax.set_yticks(y_ticks)
#     ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f"{val:.0f}"))
#     ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

#     ax.set_title(r"$\phi = " + f"{phi}$", fontsize=FONT_SIZE_TITLE, pad=7)
#     ax.set_xlabel(r"$x~\mathrm{(mm)}$", fontsize=FONT_SIZE_AXIS_LABEL)
#     if ax_idx == 0:
#         ax.set_ylabel(r"Temperature $\mathrm{(^{\circ}C)}$",
#                       fontsize=FONT_SIZE_AXIS_LABEL)

#     ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
#     ax.tick_params(which="both", direction="in", top=True, right=True, length=4)
#     ax.tick_params(which="minor", length=2)
#     # ax.grid(which="major", ls="--", lw=0.4, alpha=0.45)
#     ax.spines[["top", "right"]].set_linewidth(0.8)

#     # ── legend ────────────────────────────────────────────────────────────────
#     ax.legend(
#               loc="upper right",
#               ncol=2,
#               fontsize=FONT_SIZE_LEGEND,
#               frameon=False,
#               handlelength=1.6,
#               handletextpad=0.5,
#               columnspacing=0.8,
#               borderpad=0.5)

# # fig.suptitle(
# #     r"Spanwise-Averaged Temperature --- Effusion Coupon (Compound Angle)",
# #     fontsize=FONT_SIZE_TITLE, y=0.97
# # )

# # ── save ──────────────────────────────────────────────────────────────────────
# os.makedirs(OUT_DIR, exist_ok=True)
# fig.savefig(OUT_FILE, format="svg", bbox_inches="tight")
# print(f"Figure saved -> {OUT_FILE}")
# plt.show()

In [4]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

# ── font sizes ────────────────────────────────────────────────────────────────
FONT_SIZE_AXIS_LABEL = 12
FONT_SIZE_TICK_LABEL = 13
FONT_SIZE_LEGEND     = 12
FONT_SIZE_TITLE      = 13

# ── LaTeX rendering ───────────────────────────────────────────────────────────
plt.rcParams.update({
    "text.usetex":      True,
    "font.family":      "serif",
    "font.serif":       ["Computer Modern Roman"],
    "axes.labelsize":   FONT_SIZE_AXIS_LABEL,
    "xtick.labelsize":  FONT_SIZE_TICK_LABEL,
    "ytick.labelsize":  FONT_SIZE_TICK_LABEL,
    "legend.fontsize":  FONT_SIZE_LEGEND,
    "axes.titlesize":   FONT_SIZE_TITLE,
})

# ── case definitions ──────────────────────────────────────────────────────────
BASE = r"D:\FCAI\plain_coupon\infared_data"
SUB  = (r"time_average\average_temperature\half_domain_temperature"
        r"\summary_avg_std\cropped_end_30mm_centre")
CSV  = "combined_spanwise_avg_std_cropped.csv"

OUT_DIR = r"D:\FCAI\plain_coupon\data_processing\spanwise_averaging"

phis     = [0.85, 0.9, 1.0, 1.18, 1.37]
phi_tags = {"0.85": "085", "0.9": "09", "1.0": "10", "1.18": "118", "1.37": "137"}
crs      = [500, 1400, 2300]

COLORS = ["#1f77b4", "#d62728", "#2ca02c"]

# ── helpers ───────────────────────────────────────────────────────────────────
def load_csv(cr, phi_str):
    folder = f"cr_{cr}_phi_{phi_tags[phi_str]}"
    path = os.path.join(BASE, folder, SUB, CSV)
    if not os.path.isfile(path):
        print(f"  [WARNING] not found: {path}")
        return None
    df = pd.read_csv(path)
    df.columns = ["x", "avg", "std"]
    return df

def detect_jumps(x, threshold_factor=5):
    dx = np.diff(x)
    med = np.median(dx)
    return np.concatenate(([False], dx > threshold_factor * med))

def split_segments(df):
    x = df["x"].values
    jump = detect_jumps(x)
    segments, start = [], 0
    for i in range(1, len(x)):
        if jump[i]:
            segments.append(df.iloc[start:i].copy())
            start = i
    segments.append(df.iloc[start:].copy())
    return segments

def nice_ticks(y_lo, y_hi, n=6):
    raw_step  = (y_hi - y_lo) / (n - 1)
    magnitude = 10 ** np.floor(np.log10(raw_step))
    nice_step = np.ceil(raw_step / (5 * magnitude)) * 5 * magnitude
    t0    = np.floor(y_lo / nice_step) * nice_step
    ticks = t0 + np.arange(n) * nice_step
    return ticks, nice_step

# ── load all data ─────────────────────────────────────────────────────────────
data, missing = {}, []
for phi in phis:
    phi_str = str(phi)
    for cr in crs:
        df = load_csv(cr, phi_str)
        if df is None:
            missing.append((cr, phi))
        data[(cr, phi)] = df

if missing:
    print(f"\n[INFO] {len(missing)} case(s) missing -- curves will be skipped: {missing}\n")

# ── global y-range (shared across all figures) ────────────────────────────────
y_all_min, y_all_max = np.inf, -np.inf
for phi in phis:
    for cr in crs:
        df = data.get((cr, phi))
        if df is None:
            continue
        y_all_min = min(y_all_min, (df["avg"] - df["std"]).min())
        y_all_max = max(y_all_max, (df["avg"] + df["std"]).max())

y_ticks, step = nice_ticks(y_all_min, y_all_max, n=6)
Y_LIM = (y_ticks[0] - 0.3 * step, y_ticks[-1] + 0.3 * step)

# ── plotting — one figure per phi ─────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)

for phi in phis:
    phi_str  = str(phi)
    phi_tag  = phi_tags[phi_str]

    fig, ax = plt.subplots(figsize=(6, 4.5))
    fig.subplots_adjust(left=0.13, right=0.96, top=0.88, bottom=0.13)

    # ── draw curves ───────────────────────────────────────────────────────────
    for ci, cr in enumerate(crs):
        df = data.get((cr, phi))
        if df is None:
            continue

        color    = COLORS[ci]
        label    = r"$\dot{m}_c = $" + f"${cr}$" + r"$~\mathrm{slpm}$"
        segments = split_segments(df)

        for seg_i, seg in enumerate(segments):
            x   = seg["x"].values * 1e3
            avg = seg["avg"].values
            std = seg["std"].values
            lbl = label if seg_i == 0 else "_nolegend_"
            ax.plot(x, avg, color=color, lw=1.6, label=lbl)

    # ── axes ──────────────────────────────────────────────────────────────────
    ax.set_ylim(Y_LIM)
    ax.set_yticks(y_ticks)
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda val, _: f"{val:.0f}"))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())

    ax.set_title(r"$\phi = " + f"{phi}$", fontsize=FONT_SIZE_TITLE, pad=7)
    ax.set_xlabel(r"$x~\mathrm{(mm)}$", fontsize=FONT_SIZE_AXIS_LABEL)
    ax.set_ylabel(r"Temperature $\mathrm{(^{\circ}C)}$", fontsize=FONT_SIZE_AXIS_LABEL)

    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(which="both", direction="in", top=True, right=True, length=4)
    ax.tick_params(which="minor", length=2)
    ax.spines[["top", "right"]].set_linewidth(0.8)

    # ── legend ────────────────────────────────────────────────────────────────
    ax.legend(
        loc="upper right",
        ncol=2,
        fontsize=FONT_SIZE_LEGEND,
        frameon=False,
        handlelength=1.6,
        handletextpad=0.5,
        columnspacing=0.8,
        borderpad=0.5,
    )

    # ── save ──────────────────────────────────────────────────────────────────
    out_file = os.path.join(OUT_DIR, f"spanwise_temperature_phi_{phi_tag}.svg")
    fig.savefig(out_file, format="svg", bbox_inches="tight")
    print(f"Saved -> {out_file}")
    plt.close(fig)

print("\nAll figures saved.")

Saved -> D:\FCAI\plain_coupon\data_processing\spanwise_averaging\spanwise_temperature_phi_085.svg
Saved -> D:\FCAI\plain_coupon\data_processing\spanwise_averaging\spanwise_temperature_phi_09.svg
Saved -> D:\FCAI\plain_coupon\data_processing\spanwise_averaging\spanwise_temperature_phi_10.svg
Saved -> D:\FCAI\plain_coupon\data_processing\spanwise_averaging\spanwise_temperature_phi_118.svg
Saved -> D:\FCAI\plain_coupon\data_processing\spanwise_averaging\spanwise_temperature_phi_137.svg

All figures saved.


In [5]:
# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# import os

# # ── font sizes ────────────────────────────────────────────────────────────────
# FONT_SIZE_AXIS_LABEL = 14
# FONT_SIZE_TICK_LABEL = 13
# FONT_SIZE_LEGEND     = 13
# FONT_SIZE_TITLE      = 15

# # ── LaTeX rendering ───────────────────────────────────────────────────────────
# plt.rcParams.update({
#     "text.usetex":      True,
#     "font.family":      "serif",
#     "font.serif":       ["Computer Modern Roman"],
#     "axes.labelsize":   FONT_SIZE_AXIS_LABEL,
#     "xtick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "ytick.labelsize":  FONT_SIZE_TICK_LABEL,
#     "legend.fontsize":  FONT_SIZE_LEGEND,
#     "axes.titlesize":   FONT_SIZE_TITLE,
# })

# # ── case definitions ──────────────────────────────────────────────────────────
# BASE = r"D:\FCAI\effusion_coupon_CompundAngle\infared_data"
# SUB  = (r"time_average\average_temperature\half_domain_temperature"
#         r"\summary_avg_std\cropped_end_30mm_centre")
# CSV  = "combined_spanwise_avg_std_cropped.csv"

# OUT_DIR  = r"D:\FCAI\effusion_coupon_CompundAngle\data_processing\spanwise_averaging"
# OUT_FILE = os.path.join(OUT_DIR, "std_barplot.svg")

# phis     = [0.9, 1.0, 1.18]
# phi_tags = {"0.9": "09", "1.0": "10", "1.18": "118"}
# crs      = [500, 625, 860, 1125]

# # one colour per cooling rate, consistent with the temperature line plots
# COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e"]

# # ── helper ────────────────────────────────────────────────────────────────────
# def load_csv(cr, phi_str):
#     folder = f"cr_{cr}_phi_{phi_tags[phi_str]}"
#     path = os.path.join(BASE, folder, SUB, CSV)
#     if not os.path.isfile(path):
#         print(f"  [WARNING] not found: {path}")
#         return None
#     df = pd.read_csv(path)
#     df.columns = ["x", "avg", "std"]
#     return df

# # ── collect spatially-averaged std for every (phi, cr) ───────────────────────
# # shape: (n_phis, n_crs)  — NaN where file is missing
# mean_std = np.full((len(phis), len(crs)), np.nan)

# for i, phi in enumerate(phis):
#     for j, cr in enumerate(crs):
#         df = load_csv(cr, str(phi))
#         if df is not None:
#             mean_std[i, j] = df["std"].mean()

# # ── plotting ──────────────────────────────────────────────────────────────────
# n_phis = len(phis)
# n_crs  = len(crs)

# # bar layout: groups = phi values, within each group one bar per cr
# bar_width   = 0.18
# group_gap   = 0.1                          # extra space between phi groups
# group_width = n_crs * bar_width + group_gap
# x_centers   = np.arange(n_phis) * group_width   # centre of each phi group

# fig, ax = plt.subplots(figsize=(9, 5))
# fig.subplots_adjust(left=0.10, right=0.97, top=0.93, bottom=0.12)

# for j, cr in enumerate(crs):
#     # offset each cr bar symmetrically around the group centre
#     offsets = (np.arange(n_crs) - (n_crs - 1) / 2.0) * bar_width
#     x_pos   = x_centers + offsets[j]
#     vals    = mean_std[:, j]

#     ax.bar(x_pos, vals,
#            width=bar_width * 0.9,          # slight gap between bars
#            color=COLORS[j],
#            alpha=0.85,
#            label=r"$\dot{m}_c = $" + f"${cr}$" + r"$~\mathrm{slpm}$")

# # ── axes formatting ───────────────────────────────────────────────────────────
# phi_labels = [r"$\phi = 0.9$", r"$\phi = 1.0$", r"$\phi = 1.18$"]
# ax.set_xticks(x_centers)
# ax.set_xticklabels(phi_labels, fontsize=FONT_SIZE_TICK_LABEL)

# ax.set_ylabel(r"Mean spanwise $\sigma$ $\mathrm{(^{\circ}C)}$",
#               fontsize=FONT_SIZE_AXIS_LABEL)
# ax.set_xlabel(r"Equivalence ratio $\phi$", fontsize=FONT_SIZE_AXIS_LABEL)

# ax.tick_params(which="both", direction="in", top=True, right=True, length=4)
# ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
# ax.tick_params(which="minor", length=2)

# ax.set_xlim(x_centers[0] - group_width * 0.55,
#             x_centers[-1] + group_width * 0.55)
# ax.set_ylim(bottom=0)

# ax.legend(loc="upper right",
#           ncol=2,
#           fontsize=FONT_SIZE_LEGEND,
#           frameon=False,
#           handlelength=1.6,
#           handletextpad=0.5,
#           columnspacing=0.8)

# # ── save ──────────────────────────────────────────────────────────────────────
# os.makedirs(OUT_DIR, exist_ok=True)
# fig.savefig(OUT_FILE, format="svg", bbox_inches="tight")
# print(f"Figure saved -> {OUT_FILE}")
# plt.show()